In [ ]:
# import libraries
import pandas as pd

import re
from nltk.corpus import stopwords

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report


from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from sklearn.neighbors import KNeighborsClassifier


### Step 4: Load and Explore Data in Python

In [56]:
# load data
train = pd.read_csv("../data/Train.csv")
test = pd.read_csv("../data/Test.csv")

# View first few rows
print(train.head())

   tweet_id                                          safe_text  label  \
0  CL1KWCMY  Me &amp; The Big Homie meanboy3000 #MEANBOY #M...    0.0   
1  E3303EME  I'm 100% thinking of devoting my career to pro...    1.0   
2  M4IVFSMS  #whatcausesautism VACCINES, DO NOT VACCINATE Y...   -1.0   
3  1DR6ROZ4  I mean if they immunize my kid with something ...   -1.0   
4  J77ENIIE  Thanks to <user> Catch me performing at La Nui...    0.0   

   agreement  
0        1.0  
1        1.0  
2        1.0  
3        1.0  
4        1.0  


In [ ]:
def handle_missing_values_same_columns(train):
    # Drop rows with missing label 
    train = train.dropna(subset=['label'])
    
    # Convert label to int
    train['label'] = train['label'].astype(int)
    
    # Fill missing agreement with train median
    agreement_median = train['agreement'].median()
    train['agreement'] = train['agreement'].fillna(agreement_median)
    
    return train

In [58]:
train = handle_missing_values_same_columns(train)

In [59]:
print("Train missing values:\n", train.isnull().sum())
print("Test missing values:\n", test.isnull().sum())

Train missing values:
 tweet_id     0
safe_text    0
label        0
agreement    0
dtype: int64
Test missing values:
 tweet_id     0
safe_text    1
dtype: int64


In [ ]:
# Assign label 
sentiment = train['label']

# Check label distribution
print(sentiment.value_counts())

label
 0    4909
 1    4053
-1    1038
Name: count, dtype: int64


### Step 5: Clean and Prepare the Data

In [ ]:
import nltk
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def clean_text(text):
    if pd.isna(text):   
        return ""
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = ' '.join(word for word in text.split() if word not in stop_words)
    return text

train['clean_text'] = train['safe_text'].apply(clean_text)
test['clean_text'] = test['safe_text'].apply(clean_text)

print(f"Train shape: {train.shape}, Test shape: {test.shape}")
print(train[['safe_text', 'clean_text']].head())

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\kinut\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Train shape: (10000, 5), Test shape: (5177, 3)
                                           safe_text  \
0  Me &amp; The Big Homie meanboy3000 #MEANBOY #M...   
1  I'm 100% thinking of devoting my career to pro...   
2  #whatcausesautism VACCINES, DO NOT VACCINATE Y...   
3  I mean if they immunize my kid with something ...   
4  Thanks to <user> Catch me performing at La Nui...   

                                          clean_text  
0  amp big homie meanboy meanboy mb mbs mmr stegm...  
1  im thinking devoting career proving autism isn...  
2          whatcausesautism vaccines vaccinate child  
3  mean immunize kid something wont secretly kill...  
4  thanks user catch performing la nuit nyc st av...  


### Step 6: Convert Text to Numerical Data

In [62]:
vectorizer = TfidfVectorizer(max_features=5000)
X_train = vectorizer.fit_transform(train['clean_text'])
X_test = vectorizer.transform(test['clean_text'])

y_train = sentiment

### Step 7: Train a Classification Model

In [63]:
def train_and_evaluate(model, X_train, y_train):
    """
    Fits the model on training data and prints classification report.
    """
    model.fit(X_train, y_train)
    y_pred = model.predict(X_train)
    print(f"Model: {model.__class__.__name__}")
    print(classification_report(y_train, y_pred))
    print("-"*60)

In [ ]:
# Logistic Regression
lr = LogisticRegression(max_iter=200)
train_and_evaluate(lr, X_train, y_train)

# Random Forest Classifier
rf = RandomForestClassifier(n_estimators=100, random_state=42)
train_and_evaluate(rf, X_train, y_train)

# fitting KNN
knn = KNeighborsClassifier(n_neighbors=5)
train_and_evaluate(knn, X_train, y_train)


Model: LogisticRegression
              precision    recall  f1-score   support

          -1       0.88      0.38      0.53      1038
           0       0.85      0.89      0.87      4909
           1       0.79      0.87      0.82      4053

    accuracy                           0.82     10000
   macro avg       0.84      0.71      0.74     10000
weighted avg       0.83      0.82      0.82     10000

------------------------------------------------------------
Model: RandomForestClassifier
              precision    recall  f1-score   support

          -1       1.00      0.99      0.99      1038
           0       0.99      1.00      0.99      4909
           1       0.99      0.99      0.99      4053

    accuracy                           0.99     10000
   macro avg       1.00      0.99      0.99     10000
weighted avg       0.99      0.99      0.99     10000

------------------------------------------------------------


### Step 8: Create Final Predictions

In [67]:
# Predict on test set
test_predictions = rf.predict(X_test)

# Prepare submission file
submission = pd.DataFrame({
    'tweet_id': test['tweet_id'],
    'sentiment': test_predictions
})

submission.to_csv("submission.csv", index=False)
print("Submission file created successfully!")

Submission file created successfully!
